In [3]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
MODEL_API_VERSION=os.getenv('MODEL_API_VERSION')
if AZURE_OPENAI_API_KEY:
    print(AZURE_OPENAI_API_KEY[:10])
else:
    print("AZURE_OPENAI_API_KEY가 설정되지 않았습니다.")
print(MODEL_NAME, MODEL_API_VERSION)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    langsmith_key = os.getenv('LANGSMITH_API_KEY')
    if langsmith_key and len(langsmith_key) > 0:
        print('랭스미스로 추적 중입니다 :', langsmith_key[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
gpt-5-nano 2025-01-01-preview


In [5]:
import os
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment=os.environ["MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    openai_api_version=os.environ["MODEL_API_VERSION"],
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
print(llm.invoke("안녕!"))

from langchain_openai import AzureOpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    model=os.environ["EMB_MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    azure_deployment=os.environ["EMB_MODEL_NAME"],
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
print(embeddings.embed_query("안녕!")[:5])

content='안녕! 반가워요. 뭘 도와드릴까요? 몇 가지 예시를 들면:\n- 정보 검색이나 설명이 필요할 때\n- 글쓰기나 편집 도움\n- 코딩 문제 해결\n- 공부 계획이나 핵심 정리\n- 여행지 정보나 맛집 추천\n- 번역이나 문장 다듬기\n- 가벼운 대화나 아이디어 브레인스토밍\n\n원하는 주제를 말해 주시면 바로 도와드릴게요.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 504, 'prompt_tokens': 9, 'total_tokens': 513, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DCpA2dxsKE9jp6aYZKdITGGrSACRp', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severit

# Agentic RAG를 이용해 retreive를 최적화하기

아래 링크는 IBM에서 작성한 ai agent 문서입니다. 중요 내용을 같이 읽어봅시다.

https://www.ibm.com/kr-ko/think/topics/agentic-rag#763338456

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/pdf/InternetServiceToU.pdf", mode="page") #<-- mode="page"로 하면 페이지 단위로 나눠서 로드. 하나의 페이지가 하나의 document가 됨
docs = loader.load()
print(f"문서 개수: {len(docs)}")
for doc in docs:
    doc.metadata["producer"] = ""
    doc.metadata["creator"] = ""
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\n", "\n", "  ", " "]
)
splits = splitter.split_documents(docs)
from langchain_community.vectorstores import FAISS
vectordb = FAISS.from_documents(splits,embeddings)

문서 개수: 232


In [7]:
from langchain.tools import tool

@tool(response_format="content")
def retrieve_context(query: str):
    """질문에 대한 정보를 검색하는 도구
    
    ## return data는 아래 두 가지
    - serialized : 검색된 문서의 내용의 직렬화된 문자열
    - retrieved_docs : 검색된 문서의 목록(document의 list)"""
    
    retrieved_docs = vectordb.similarity_search(query, k=5)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized

result = retrieve_context.invoke("무료 이용기간은 얼마나 되나요?")
print(result)

Source: {'producer': '', 'creator': '', 'creationdate': '2025-09-03T15:58:56+09:00', 'source': '../data/pdf/InternetServiceToU.pdf', 'file_path': '../data/pdf/InternetServiceToU.pdf', 'total_pages': 232, 'format': 'PDF 1.7', 'title': '', 'author': '박현수(상품시너지팀)\x00', 'subject': '', 'keywords': '', 'moddate': '2025-09-03T15:58:56+09:00', 'trapped': '', 'modDate': "D:20250903155856+09'00'", 'creationDate': "D:20250903155856+09'00'", 'page': 182}
Content: - 산정식: 이용기간x(계약기간 할인금액-사용기간 할인금액) 
계약기간 
할인금액 
사용기간 할인금액 
1 년미만 
1 년이상 ~ 2 년미만 
2 년이상 ~ 3 년미만 
5,500 원 
- 
1,650 원 
3,300 원 
※ 월정액 5,500원 추가 시, olleh internet 스페셜 급의 속도 제공 
※ 이용약관 제 17 조제 5 항에 의하여 보증금을 납부하는 경우 납부 금액은 60,000원이며, 청약 시 선납하여야 됩니다.

Source: {'producer': '', 'creator': '', 'creationdate': '2025-09-03T15:58:56+09:00', 'source': '../data/pdf/InternetServiceToU.pdf', 'file_path': '../data/pdf/InternetServiceToU.pdf', 'total_pages': 232, 'format': 'PDF 1.7', 'title': '', 'author': '박현수(상품시너지팀)\x00', 'subject': '', 'keywords': '', '

In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tools = [retrieve_context]
prompt = (
    "너는 통신사 고객상담을 위한 이용약관 문서 기반 QA 어시스턴트다. 한국어로 간단하게 답하고 아래 규칙을 지켜라.\n"
    "정보의 검색은 retrieve_context 도구를 이용한다. 사용자의 질문으로부터 검색어를 정제하고 이를 답변 상단에 언급한다.\n"
     "1) 제공된 컨텍스트(표/본문)에서만 근거를 찾아 답한다.\n"
     "2) 컨택스트 내에 해당하는 정보가 없으면 모른다고 말한다. 추측 금지.\n"
     "3) 숫자/조건/예외는 정확히 인용하라.\n"
     "4) 표 내용이라면 행/열 헤더를 함께 언급해 맥락을 명확히 한다.\n"
)
agent = create_agent(llm, tools, system_prompt=prompt, checkpointer=InMemorySaver())

In [9]:
def agent_run(question, mode = 'invoke'):
    cfg = {'configurable': {'thread_id': 'user-1'}}
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=cfg)
    return result.get("messages", [])[-1].content

In [10]:
print(agent_run("Giga wifi 요금제에 대해서 설명해줘!"))

검색어: Giga wifi 요금제

다음은 주어진 컨텍스트에 따른 GiGA WiFi 요금제 관련 내용 요약입니다.

- 선납제(장비 임대료 선납)
  - 원칙: 인터넷 AP 장비 임대료를 선납하는 경우 할인
  - 선납금(3년 약정 고객 한함)
  - 구분 및 선납금:
    - AP: 홈허브스페셜, WiFi home — 33,000원
    - GiGA WiFi home — 99,000원
    - GiGA WiFi home plus — 49,500원
    - GiGA WiFi Premium — 165,000원
    - GiGA WiFi premium plus — 82,500원
    - GiGA WiFi premium 2.4(6E) — 132,000원
    - 단, ‘24년 5월 31일 이전 가입자는 231,000원
    - GiGA WiFi premium 4.8 (다만 라인이 불완전해 4.8의 선납금은 문맥상 독립적으로 기입되었는지 불확실) 

- 3년 약정 시 월 임대료 할인(이용 기간 3년 약정 대상)
  - 대상: GiGA WiFi home, GiGA WiFi premium AP 신규 3년 약정 고객
  - 대상 요금: 3년 약정 월 임대료
  - 할인 조건 및 내역(요약):
    - 표에 따르면 인터넷/모바일/TV와 UHD STB 여부에 따라 할인액이 다르게 적용되는 형태로 보이며, “UHD STB 이용 시”의 할인은 0원으로 표시되고, 일반 조건에서 월 1,100원이 할인되는 경우도 있음
    - 구체적 구성은 표의 행/열 구분에 따라 결정되며, 부가세 포함 금액으로 기재

- 2022년 프로모션(기간: 2022-07-01 ~ 2022-12-31)
  - GiGA WiFi Premium 2.4(6E) 신규가입 시 인터넷 이용요금 할인(대상: GiGA WiFi Premium 2.4(6E))
    - 대상 인터넷상품: 인터넷 베이직, 인터넷 에센스
    - 할인액: 인터넷 베이직 2,200원, 인터넷 에센스 4,400원
  - GiGA

In [11]:
print(agent_run("그 중 프리미엄의 특징은?"))

검색어: GiGA WiFi Premium

다음은 컨텍스트에서 확인 가능한 프리미엄(GiGA WiFi Premium) 특징 요약입니다.

- 프리미엄은 GiGA WiFi AP 중 상위 모델로 분류되는 단말임.
- 선납제(임대료 선납) 적용 대상이며, 선납금은 다음과 같음
  - GiGA WiFi Premium — 165,000원
  - GiGA WiFi Premium 2.4(6E) — 132,000원
  - 단, 24년 5월 31일 이전 가입자는 231,000원
- 3년 약정 시 월 임대료 할인 대상에 포함되며 대상은 “GiGA WiFi home, GiGA WiFi premium AP 신규 3년 약정 고객”.
  - 표로 할인 조건이 제시되며, 부가세 포함
  - 구분: 필수 / 선택(필수 회선에 결합 시)
  - UHD STB 이용 시 할인: 0원
  - 그 외의 경우 할인: 월 1,100원
- 2022년 프로모션 예시
  - GiGA WiFi Premium 2.4(6E) 신규 가입 시 인터넷 이용요금 할인
    - 대상: 인터넷 프리미엄플러스 가입자 중 GiGA WiFi Premium 2.4(6E) 신규 신청
    - 신청기간: 2022년 7월 1일 ~ 2022년 12월 31일
    - 할인액: 인터넷 베이직 2,200원, 인터넷 에센스 4,400원
  - GiGA WiFi Premium 4.8 신규 가입 시 인터넷 이용요금 할인
    - 대상: 인터넷 에센스, 인터넷 프리미엄
    - 신청기간: 2022년 10월 1일 ~ 2023년 3월 31일
    - 할인액: 인터넷 에센스 5,500원, 인터넷 프리미엄 6,600원
  - 기타 주의: GiGA WiFi Premium 2.4(6E) 재고 소진 시 프로모션 조기 종료 가능

필요하면 위 항목 중 특정 수치나 조건을 더 자세히 정리해 드릴게요.
